In [1]:
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
import os
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from IPython.display import clear_output
from defs import play_avi
import cv2


In [ ]:
import mediapipe as mp

In [3]:
load_dotenv()
hf_token = os.getenv("hug_f_token")


ds = load_dataset("kyegorov/mcd_rppg", streaming=True, split="train", token=hf_token)

ds

Resolving data files:   0%|          | 0/12002 [00:00<?, ?it/s]

IterableDataset({
    features: Unknown,
    num_shards: 7201
})

In [ ]:
from huggingface_hub import hf_hub_download
import pandas as pd

# 1. Download the master mapping file
print("Downloading db.csv...")
csv_path = hf_hub_download(repo_id="kyegorov/mcd_rppg", repo_type="dataset", filename="db.csv")
df = pd.read_csv(csv_path)

print(df.head())


target_video = df['video'][100]

print(f"Downloading {target_video}...")
video_path = hf_hub_download(repo_id="kyegorov/mcd_rppg", repo_type="dataset", filename=target_video)

video_path

   patient_id  weight  height        bmi   age sex  upper_ap  lower_ap  \
0        1020    55.0   170.0  19.031142  23.0   F     113.0      78.0   
1        1020    55.0   170.0  19.031142  23.0   F     113.0      78.0   
2        1020    55.0   170.0  19.031142  23.0   F     113.0      78.0   
3        1020    55.0   170.0  19.031142  23.0   F     105.0      78.0   
4        1020    55.0   170.0  19.031142  23.0   F     105.0      78.0   

   saturation  temperature  ...  pulse  stress    step        camera   view  \
0        98.0         36.6  ...  100.0     4.0   after  FullHDwebcam  front   
1        98.0         36.6  ...  100.0     4.0   after      USBVideo   left   
2        98.0         36.6  ...  100.0     4.0   after   IriunWebcam  right   
3        99.0         36.6  ...   83.0     4.0  before  FullHDwebcam  front   
4        99.0         36.6  ...   83.0     4.0  before      USBVideo   left   

                    ecg                 ppg  \
0   ecg/1020_after.json   ppg/102

In [ ]:
play_avi(video_path)

In [ ]:
import cv2
import mediapipe as mp
from IPython.display import display, clear_output
from PIL import Image

# 1. MediaPipe Setup
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_face_mesh = mp.solutions.face_mesh

face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# 2. Video-Pfad eintragen (ersetze dies mit deinem Dateinamen)

cap = cv2.VideoCapture(video_path)

# Wir nutzen einen try-except Block, damit du die Zelle einfach stoppen kannst, 
# ohne dass die Videodatei blockiert bleibt.
try:
    while cap.isOpened():
        success, image = cap.read()
        if not success:
            print("Video zu Ende gelesen.")
            break

        # Umwandlung für MediaPipe (OpenCV ist BGR, MediaPipe will RGB)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Das Bild durch das Modell schicken
        results = face_mesh.process(image_rgb)

        # Die Punkte auf das RGB-Bild zeichnen
        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                mp_drawing.draw_landmarks(
                    image=image_rgb, # Wir zeichnen direkt auf das RGB-Bild
                    landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_TESSELATION,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_tesselation_style()
                )

        # --- JUPYTER NOTEBOOK MAGIE ---
        # 1. Numpy-Array in ein anzeigbares Bild-Objekt umwandeln
        img_pil = Image.fromarray(image_rgb)
        
        # 2. Das alte Bild löschen (wait=True verhindert Flackern)
        clear_output(wait=True)
        
        # 3. Das neue Bild in der Zelle anzeigen
        display(img_pil)

except KeyboardInterrupt:
    print("Vorschau manuell vom Nutzer abgebrochen.")
    
finally:
    # Aufräumen: Datei wieder freigeben
    cap.release()

In [5]:
import cv2
import mediapipe as mp
import numpy as np
from tqdm.notebook import tqdm
FOREHEAD_NODES = [10, 151, 67, 109, 108, 107, 297, 338, 336, 337]
# Cheeks: fleshy mid-cheek, off the nose and away from mouth corners
CHEEK_NODES    = [50, 205, 187, 123, 116, 280, 425, 411, 352, 345]
SELECTED_NODES = FOREHEAD_NODES + CHEEK_NODES
N_NODES = len(SELECTED_NODES)

face_mesh = mp.solutions.face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)



Z_OCCLUSION_MARGIN = 0.07


def extract_frame_features(frame, out):
    """Fills out (N_NODES, 4) in-place with [x, y, avg_green, valid] per node.

    valid = 1 if the landmark is on the camera-facing side AND the patch fits in frame,
    valid = 0 (and other channels zeroed) otherwise.
    """
    h, w, _ = frame.shape
    results = face_mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    out.fill(0)
    if not results.multi_face_landmarks:
        return  # no face -> all zeros, valid=0 everywhere

    landmarks = results.multi_face_landmarks[0].landmark

    # Occlusion via z: far-side landmarks have larger z than the front-most node.
    zs = np.array([landmarks[idx].z for idx in SELECTED_NODES], dtype=np.float32)
    z_front = zs.min()
    occluded = zs > z_front + Z_OCCLUSION_MARGIN

    for i, idx in enumerate(SELECTED_NODES):
        if occluded[i]:
            continue  # leave as zeros, valid stays 0
        pt = landmarks[idx]
        px, py = int(pt.x * w), int(pt.y * h)
        if not (2 <= px < w - 2 and 2 <= py < h - 2):
            continue  # patch off-frame -> treat as invalid
        out[i, 0] = pt.x
        out[i, 1] = pt.y
        out[i, 2] = frame[py-2:py+3, px-2:px+3, 1].mean()
        out[i, 3] = 1.0  # valid


def extract_graph_features(video_path):
    output_name = os.path.splitext(os.path.basename(video_path))[0]
    save_dir = os.path.join(os.getcwd(), "final_matrix_video")
    save_path = os.path.join(save_dir, output_name + ".npy")
    if os.path.exists(save_path):
        return "already done"

    os.makedirs(save_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Pre-allocate when total_frames is known; otherwise grow in chunks.
    if total_frames > 0:
        buf = np.empty((total_frames, N_NODES, 4), dtype=np.float32)
        idx = 0
        pbar = tqdm(total=total_frames, desc=output_name, unit="frame")
        while True:
            success, frame = cap.read()
            if not success:
                break
            if idx >= total_frames:
                buf = np.concatenate([buf, np.empty((256, N_NODES, 4), dtype=np.float32)], axis=0)
            extract_frame_features(frame, buf[idx])
            idx += 1
            pbar.update(1)
        pbar.close()
        final_matrix = buf[:idx]
    else:
        chunks = []
        chunk = np.empty((512, N_NODES, 4), dtype=np.float32)
        idx = 0
        pbar = tqdm(desc=output_name, unit="frame")
        while True:
            success, frame = cap.read()
            if not success:
                break
            if idx == chunk.shape[0]:
                chunks.append(chunk)
                chunk = np.empty((512, N_NODES, 4), dtype=np.float32)
                idx = 0
            extract_frame_features(frame, chunk[idx])
            idx += 1
            pbar.update(1)
        pbar.close()
        chunks.append(chunk[:idx])
        final_matrix = np.concatenate(chunks, axis=0)

    cap.release()
    np.save(save_path, final_matrix)
    print(f"Saved: {save_path}  Shape: {final_matrix.shape}")
    return final_matrix




I0000 00:00:1777450944.608634   13391 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [ ]:
# Visual sanity check: show ONLY the landmarks that pass the same valid/occlusion
# checks used by extract_frame_features. Tweak Z_OCCLUSION_MARGIN above and re-run.
import cv2
from IPython.display import display, clear_output
from PIL import Image

PATCH_HALF = 2  # matches frame[py-2:py+3, px-2:px+3] in extract_frame_features


def preview_roi(video_path, max_frames=300, stride=2, show_occluded=True):
    """show_occluded=True draws masked-out points as a faint red X for debugging.
    Set False to see only what the GNN will actually receive."""
    cap = cv2.VideoCapture(video_path)
    shown = 0
    try:
        while cap.isOpened() and shown < max_frames:
            success, frame = cap.read()
            if not success:
                break
            if shown % stride != 0:
                shown += 1
                continue

            h, w, _ = frame.shape
            results = face_mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            vis = frame.copy()

            if results.multi_face_landmarks:
                lms = results.multi_face_landmarks[0].landmark

                # Same occlusion logic as extract_frame_features
                zs = np.array([lms[idx].z for idx in SELECTED_NODES], dtype=np.float32)
                z_front = zs.min()
                occluded = zs > z_front + Z_OCCLUSION_MARGIN

                n_valid = 0
                for i, idx in enumerate(SELECTED_NODES):
                    px = int(lms[idx].x * w)
                    py = int(lms[idx].y * h)
                    in_frame = 2 <= px < w - 2 and 2 <= py < h - 2
                    is_valid = (not occluded[i]) and in_frame

                    if is_valid:
                        color = (0, 255, 0) if i < len(FOREHEAD_NODES) else (0, 200, 255)
                        cv2.rectangle(vis,
                                      (px - PATCH_HALF, py - PATCH_HALF),
                                      (px + PATCH_HALF, py + PATCH_HALF),
                                      color, 1)
                        cv2.putText(vis, str(idx), (px + 4, py - 4),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.3, color, 1, cv2.LINE_AA)
                        n_valid += 1
                    elif show_occluded:
                        # Faint red X for masked nodes (occluded or off-frame)
                        cv2.line(vis, (px - 3, py - 3), (px + 3, py + 3), (0, 0, 180), 1)
                        cv2.line(vis, (px - 3, py + 3), (px + 3, py - 3), (0, 0, 180), 1)

                cv2.putText(vis,
                            f"valid {n_valid}/{N_NODES}  margin={Z_OCCLUSION_MARGIN:.2f}",
                            (10, 25),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
            else:
                cv2.putText(vis, "NO FACE DETECTED", (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

            clear_output(wait=True)
            display(Image.fromarray(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)))
            shown += 1
    except KeyboardInterrupt:
        print("Stopped by user.")
    finally:
        cap.release()


preview_roi(video_path, max_frames=300, stride=2, show_occluded=True)


In [ ]:
import threading
import queue

def download_extract_video():
    videos = df["video"].drop_duplicates().tolist()
    save_dir = os.path.join(os.getcwd(), "final_matrix_video")
    os.makedirs(save_dir, exist_ok=True)

    # Skip already-processed up front so we don't queue them.
    todo = []
    for v in videos:
        out_name = os.path.splitext(os.path.basename(v))[0] + ".npy"
        if not os.path.exists(os.path.join(save_dir, out_name)):
            todo.append(v)
    print(f"{len(todo)} videos to process ({len(videos) - len(todo)} already done)")

    q = queue.Queue(maxsize=2)  # 1 ready + 1 being downloaded
    SENTINEL = object()

    def producer():
        for v in todo:
            try:
                path = hf_hub_download(
                    repo_id="kyegorov/mcd_rppg", repo_type="dataset", filename=v
                )
                q.put(path)
            except Exception as e:
                print(f"  download error {v}: {e}")
        q.put(SENTINEL)

    t = threading.Thread(target=producer, daemon=True)
    t.start()

    for i in range(len(todo)):
        path = q.get()
        if path is SENTINEL:
            break
        try:
            print(f"[{i+1}/{len(todo)}] extracting {os.path.basename(path)}")
            extract_graph_features(path)
        except Exception as e:
            print(f"  extract error {path}: {e}")
        finally:
            try:
                os.remove(path)  # free disk
            except OSError:
                pass

    t.join()


download_extract_video()